<a href="https://colab.research.google.com/drive/1fT11JsthwBe3mRlwqSwhrnlJpdTvEl6B?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

###	➕ What is Query Expansion/Transformation RAG?

Query Expansion is a technique used to improve the effectiveness of information retrieval systems by reformulating or augmenting the original query. The goal is to improve the recall of relevant documents by including related terms or concepts that might not have been explicitly mentioned in the original query.

### 🔧 Query Transformation RAG Implementation:

1. **Query Transformation:** We use the LLM to generate 3 alternative versions of the original query.
2. **Multi-Query Retrieval:** We retrieve documents using both the original query and the transformed queries.
3. **Deduplication:** We remove duplicate documents from the retrieved set.
4. **Response Generation:** Using the combined context from all retrieved documents, we generate a final response to the original query.

# ⚙️ Setup

1. **[LLM](https://groq.com/):** Groq's free Open source LLM endpoints([Groq API Key](https://console.groq.com/keys))
2. **[Vector Store](https://www.pinecone.io/learn/vector-database/):** [ChromaDB](https://www.trychroma.com/)
3. **[Embedding Model](https://qdrant.tech/articles/what-are-embeddings/):** [nomic-embed-text-v1.5](https://www.nomic.ai/blog/posts/nomic-embed-text-v1)
4. **[LLM Framework](https://python.langchain.com/v0.2/docs/introduction/):** LangChain
5. **[Huggingface API Key](https://huggingface.co/settings/tokens)**


# Install required libraries

In [ ]:
!pip install -q -U \
     langchain==0.3.19 \
     langchain-groq==0.2.4 \
     langchain-chroma==0.2.2 \
     langchain-community==0.3.18 \
     langchain-huggingface==0.1.2 \
     einops==0.8.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 459.1/459.1 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

# Import related libraries related to Langchain, HuggingfaceEmbedding

In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import ChatPromptTemplate
from langchain.document_loaders import WebBaseLoader

In [ ]:
import os
import getpass

#### Provide a Groq API key. You can create one to access free open-source models at the following link.

[Groq API Creation Link](https://console.groq.com/keys)




In [ ]:
os.environ["GROQ_API_KEY"] = getpass.getpass()

··········


### Provide Huggingface API Key. You can create Huggingface API key at following link

[Huggingface API Creation Link](https://huggingface.co/settings/tokens)




In [ ]:
os.environ["HF_TOKEN"] = getpass.getpass()

··········


### Step 1: Load and preprocess data code

In [ ]:
def load_and_process_data(url):
    # Load data from web
    loader = WebBaseLoader(url)
    data = loader.load()

    # Split text into chunks (Experiment with Chunk Size and Chunk Overlap to get optimal chunking)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = text_splitter.split_documents(data)

    return chunks

### Step 2: Create multiple vector stores code

In [ ]:
def create_vector_store(chunks):
  embeddings = HuggingFaceEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5", model_kwargs = {'trust_remote_code': True})
  vectorstore = Chroma.from_documents(chunks, embeddings)
  return vectorstore

### Step 3: Query Transformation-RAG related code

1. **Query Transformation:** We use the LLM to generate 3 alternative versions of the original query.
2. **Multi-Query Retrieval:** We retrieve documents using both the original query and the transformed queries.
3. **Deduplication:** We remove duplicate documents from the retrieved set.
4. **Response Generation:** Using the combined context from all retrieved documents, we generate a final response to the original query.

In [ ]:
def query_transformation_rag(original_query, vectorstore, llm):
    # Query Transformation
    transform_prompt = ChatPromptTemplate.from_template(
        "Given the original query, generate 3 alternative versions that might improve "
        "retrieval effectiveness. Each version should capture a different aspect or "
        "use different terminology related to the original query.\n"
        "Original query: {query}\n"
        "Transformed queries (provide 3):"
    )
    transform_chain = transform_prompt | llm
    try:
        transformed_queries_response = transform_chain.invoke({"query": original_query})
        transformed_queries = transformed_queries_response.content.split('\n')
        transformed_queries = [q.split(': ')[-1].strip() for q in transformed_queries if q.strip()]
        transformed_queries = transformed_queries[:3]  # Ensure we have at most 3 queries
    except Exception as e:
        print(f"Error transforming query: {e}")
        transformed_queries = [original_query]  # Fallback to original query

    # Retrieve documents for each transformed query
    all_docs = []
    for query in [original_query] + transformed_queries:
        docs = vectorstore.similarity_search(query, k=2)
        all_docs.extend(docs)

    # Remove duplicates and combine retrieved documents
    unique_docs = list({doc.page_content: doc for doc in all_docs}.values())
    context = "\n\n".join([doc.page_content for doc in unique_docs])

    # Generate response using combined context
    response_prompt = ChatPromptTemplate.from_template(
        "You are an AI assistant tasked with answering questions based on the provided context. "
        "The context contains information retrieved using the original query and its transformed versions. "
        "Please analyze the context carefully and provide a comprehensive answer to the original question. "
        "If the context doesn't contain enough information, use your general knowledge to supplement the answer, "
        "but prioritize information from the context when available.\n\n"
        "Original Question: {original_query}\n"
        "Context:\n{context}\n\n"
        "Answer:"
    )
    response_chain = response_prompt | llm
    try:
        response = response_chain.invoke({"context": context, "original_query": original_query})
        final_answer = response.content
    except Exception as e:
        print(f"Error generating response: {e}")
        final_answer = "I apologize, but I encountered an error while generating the response."

    return {
        "original_query": original_query,
        "transformed_queries": transformed_queries,
        "final_answer": final_answer,
        "retrieved_context": context
    }

In [ ]:
pip uninstall -y numpy pandas scipy scikit-learn


Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1


In [ ]:
pip install numpy==1.26.4 pandas

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.3 MB/s eta 0:00:00
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 85.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 3.0.1 requires scikit-learn, which is not installed.
sentence-transformers 3.0.1 requires scipy, which is not installed.
stumpy 1.13.0 requires scipy>=1.10, which is not installed.
umap-learn 0.5.12 requires scikit-learn>=1.6, which is not installed.
umap-learn 0.5.12 requires scipy>=1.3.1, which is not installed.
spglm 1.1.0 requires scipy>=1.8, which is not installed.
tobler 0.14.0 requires scipy>=1.13, which is not installed.
clarabel 0.1

### Step 4: Create chunk of web data to Chroma Vector Store

In [ ]:
pip install sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 21.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.5
)

# Load and process data
url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
chunks = load_and_process_data(url)

# Create vector store
vectorstore = create_vector_store(chunks)

modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Step 5: Run Query Transformation RAG

This implementation shows the key parts of Query Transformation RAG:

1. Generation of alternative query formulations to capture different aspects or terminology
2. Retrieval using both original and transformed queries to improve coverage
3. Deduplication of retrieved documents to avoid redundancy
Generation of a comprehensive response using the expanded context

In [ ]:
# Example query
original_query = "What are the ethical considerations in AI development?"

# Run Query Transformation RAG
result = query_transformation_rag(original_query, vectorstore, llm)

print("Original Query:", result["original_query"])
print("\nTransformed Queries:")
for i, query in enumerate(result["transformed_queries"], 1):
    print(f"{i}. {query}")
print("\nFinal Answer:")
print(result["final_answer"])
print("\nRetrieved Context (first 500 characters):")
print(result["retrieved_context"][:500] + "...")